# MATH 6243 Final Project (Sandy's code)

Sandy Xu


## Further Data Cleaning

To help with class imbalance, prevent high cardinality, and help run the code without it taking a very long time, we grouped some of the categories in the features (both the predictors and target feature of force type). We also decided to drop neighborhood and primary offense as features. Later, we decided to randomly subsample from the Force Type groups since the class imbalance was still very high. So we randomly subsampled from each of the 4 groups 478 observations (to match the one with the least number of observations which was "Guns").

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, mean_squared_error
from sklearn.preprocessing import TargetEncoder, LabelEncoder

# Checking the shape of the dataset
df = pd.read_csv("cleaned_data.csv")
print("Shape:", df.shape)

# Dropping the unnamed column, Neighborhood, and PrimaryOffense
df = df.drop(columns=["Unnamed: 0", "Neighborhood", "PrimaryOffense"])

# Printing the updated feature list
print("Updated features:", df.columns.tolist())


# New Section

### *Grouping Force Type:*

In [ ]:
print("\nForceType distribution:")
print(df["ForceType"].value_counts())


There is a heavy class imbalance with about 72.05% of the force type used being bodily force. To help reduce this class imbalance, I will combine some of the categories in force type (the target variable). We decided to group the following together:

-Bodily force: "Bodily Force" and "Maximal Restraint Technique"

-Chemical irritant: "Chemical Irritant"

-Less-lethal weapons: "Less Lethal Projectile", "Baton", "Less Lethal", "Taser", "Improvised Weapon", "Police K9 Bite"

-Guns: "Gun Point Display" and "Firearm"


 This combined the force types into 4 total categories, as shown below.


In [ ]:
# Combining ForceType categories to help reduce class imbalance
force_map = {
    "Bodily Force": "Bodily force",
    "Maximal Restraint Technique": "Bodily force",

    "Chemical Irritant": "Chemical Irritant",

    "Less Lethal Projectile": "Less-lethal weapons",
    "Baton": "Less-lethal weapons",
    "Less Lethal": "Less-lethal weapons",
    "Taser": "Less-lethal weapons",
    "Improvised Weapon": "Less-lethal weapons",
    "Police K9 Bite": "Less-lethal weapons",

    "Gun Point Display": "Guns",
    "Firearm": "Guns"
}

df["ForceGroup"] = df["ForceType"].map(force_map)
print(df["ForceGroup"].value_counts())


### *Grouping Race:*

In [ ]:
print(df["Race"].value_counts())


In [ ]:
df['Race'] = df['Race'].replace('Other / Mixed Race', 'other')
df['Race'] = df['Race'].replace('Pacific Islander', 'other')
df['Race'] = df['Race'].replace('Asian', 'other')
df['Race'] = df['Race'].replace('Unknown', 'other')
df = df[df['Race'] != 'not recorded']
print(df['Race'].value_counts())


### *Grouping Type of Resistance:*

In [ ]:
print(df["TypeOfResistance"].value_counts())


In [ ]:
# Grouping together types of resistance (and combining the ones that are the same)
df["TypeOfResistance"] = df["TypeOfResistance"].str.strip().str.lower()
# since some of the types of resistance are categorized as different but just have different capitalization, str.lower() makes them all lowercase
# str.strip() removes different empty spaces or characters from the beginning and end of the categories

resistance_map = {
    "assaulted officer":       "Assaulting",
    "assaulting police horse": "Assaulting",
    "assaulted police horse":  "Assaulting",
    "assaulting police k9":    "Assaulting",
    "assaulted police k9":     "Assaulting",

    "fled on foot":            "Fleeing",
    "fled in a vehicle":       "Fleeing",
    "fled in vehicle":         "Fleeing",

    "commission of crime":     "Commission of Crime",
    "commission of a crime":   "Commission of Crime",

    "unspecified":             "Other/Unspecified",
    "other":                   "Other/Unspecified",

    "tensed":                  "Tensed",

    "verbal non-compliance":   "Verbal Non-Compliance"
}

df["ResistanceGroup"] = df["TypeOfResistance"].replace(resistance_map)

# Drop the 33 unknown 'x' rows
df = df[df["ResistanceGroup"] != "x"]

print(df["ResistanceGroup"].value_counts())


### *Grouping Problem Type:*

In [ ]:
# Grouping problem type (grabbed from Mishka's code!)
problem_map = {
   "fight": "Violent", "assault in progress": "Violent", "assault report only": "Violent",
   "stabbing": "Violent", "shooting": "Violent", "shooting report only": "Violent",
   "robbery of person": "Violent", "robbery dwell in progress": "Violent",
   "robbery of biz in progress": "Violent", "robbery dwlng/person rpt": "Violent",
   "robbery of biz - report": "Violent", "person with a gun": "Violent",
   "person with a weapon": "Violent", "domestic with weapons": "Violent",
   "shotspotter activation": "Violent", "sound of shots fired": "Violent",
   "holdup alarm": "Violent", "kidnapping/abduction": "Violent",


   "domestic abuse-in progress": "Domestic", "domestic abuse report only": "Domestic",
   "domestic": "Domestic", "neighbor trouble": "Domestic", "tenant trouble": "Domestic",
   "retrieve prop/dom situation": "Domestic", "threats": "Domestic",
   "crim sex conduct": "Domestic", "crim sex conduct/report": "Domestic",


   "suspicious person": "Suspicious & Disorder", "suspicious vehicle": "Suspicious & Disorder",
   "disturbance": "Suspicious & Disorder", "unwanted person": "Suspicious & Disorder",
   "customer trouble": "Suspicious & Disorder", "loud party": "Suspicious & Disorder",
   "music-loud": "Suspicious & Disorder", "drunk/intoxicated person": "Suspicious & Disorder",
   "narcotics": "Suspicious & Disorder", "indecent exposure": "Suspicious & Disorder",
   "prowler": "Suspicious & Disorder", "trespass in boarded dwell": "Suspicious & Disorder",
   "unknown trouble": "Suspicious & Disorder", "curfew violations": "Suspicious & Disorder",
   "hotrodders": "Suspicious & Disorder", "suspected prostitute": "Suspicious & Disorder",
   "attempt pick-up": "Suspicious & Disorder", "walk through a building": "Suspicious & Disorder",
   "parking problem": "Suspicious & Disorder", "truancy": "Suspicious & Disorder",
   "firecrackers": "Suspicious & Disorder", "kid trouble": "Suspicious & Disorder",


   "burglary dwlng in progress": "Property & Vehicle", "burglary biz - in progress": "Property & Vehicle",
   "burglary dwlng - report": "Property & Vehicle", "burglary business - report": "Property & Vehicle",
   "theft": "Property & Vehicle", "theft-hold one cooperative": "Property & Vehicle",
   "theft - report only": "Property & Vehicle", "auto theft in progress": "Property & Vehicle",
   "auto theft": "Property & Vehicle", "bait vehicle auto theft": "Property & Vehicle",
   "recover vehicle": "Property & Vehicle", "recover property": "Property & Vehicle",
   "motor vehicle chase": "Property & Vehicle", "chase on foot": "Property & Vehicle",
   "forgery in progress": "Property & Vehicle", "forgery report": "Property & Vehicle",
   "damage property-in progress": "Property & Vehicle", "damage property-rpt only": "Property & Vehicle",
   "traffic law enforcement": "Property & Vehicle", "property damage accident": "Property & Vehicle",
   "personal injury accident": "Property & Vehicle", "property damage/hit & run": "Property & Vehicle",
   "poss personal injury acc": "Property & Vehicle", "personal inj/hit and run": "Property & Vehicle",
   "audible business alarm": "Property & Vehicle", "audible residential alarm": "Property & Vehicle",
   "audible alarm": "Property & Vehicle", "silent alarm": "Property & Vehicle",
   "panic alarm": "Property & Vehicle",


   "emotionally disturb person": "Welfare & Emergency", "check the welfare": "Welfare & Emergency",
   "attempted suicide": "Welfare & Emergency", "person threat to jump": "Welfare & Emergency",
   "down outside-one": "Welfare & Emergency", "down outside-one w/fire": "Welfare & Emergency",
   "down outide-one w/fire": "Welfare & Emergency", "slumper": "Welfare & Emergency",
   "slumper w/fire": "Welfare & Emergency", "unconscious": "Welfare & Emergency",
   "overdose-accidental": "Welfare & Emergency", "overdose w/all": "Welfare & Emergency",
   "overdose w/alll": "Welfare & Emergency", "officer needs help": "Welfare & Emergency",
   "assist an officer": "Welfare & Emergency", "assist other agency": "Welfare & Emergency",
   "assist ems personnel": "Welfare & Emergency", "assist fire personnel": "Welfare & Emergency",
   "paramedic needs help": "Welfare & Emergency", "high risk warrant entry": "Welfare & Emergency",
   "code 3": "Welfare & Emergency", "dead person": "Welfare & Emergency",
   "missing person": "Welfare & Emergency", "lost child": "Welfare & Emergency",
   "drowning": "Welfare & Emergency", "miscellaneous": "Welfare & Emergency",
   "unknown wireless/cell phone": "Welfare & Emergency", "on site": "Welfare & Emergency",
   "transportation": "Welfare & Emergency", "directed patrol": "Welfare & Emergency",
   "assist a disabled person": "Welfare & Emergency", "pandemic emergency": "Welfare & Emergency",
   "police event": "Welfare & Emergency", "receive information": "Welfare & Emergency",
   "business check": "Welfare & Emergency", "community engagement meeting": "Welfare & Emergency",
   "suspected hazard": "Welfare & Emergency", "check hazard": "Welfare & Emergency",
   "foot beat": "Welfare & Emergency", "firefighter needs help": "Welfare & Emergency",
   "assist-public": "Welfare & Emergency", "road hazard": "Welfare & Emergency",
   "wires down": "Welfare & Emergency", "animal bite": "Welfare & Emergency",
   "animal call": "Welfare & Emergency", "aggressive dog": "Welfare & Emergency",
   "fire-report only": "Welfare & Emergency", "pi w/trapped": "Welfare & Emergency",
   "elevator emergency": "Welfare & Emergency", "diabetic problem": "Welfare & Emergency",
   "baby not breathing": "Welfare & Emergency", "test/demo ee & cad oper": "Welfare & Emergency",
   "crank 9-1-1 call": "Welfare & Emergency", "aircraft crash in city": "Welfare & Emergency",
}


df['Problem_Group'] = df['Problem'].str.strip().str.lower().map(problem_map).fillna("Welfare & Emergency")
print(df['Problem_Group'].value_counts())


### *Cleaning up Sex:*

In [ ]:
df = df[df['Sex'] != 'not recorded']
print(df['Sex'].value_counts())


## Bayesian Model

For the portion I am coding, I use a Bayesian model for analysis.

I chose the likelihood distribution for the target variable, ForceType, as normal distributions centered at 0 with a standard deviation of 1.

The data in this csv is already cleaned (thank you Celia!) so we don't need to drop any data points with null values in this code.

I encoded all of the categorical features to be integers. We can't one-hot encode for all of the features since they aren't binary. I encoded the problem group using TargetEncoder from sklearn.preprocessing since there were many problem types that were grouped and it would be difficult to one hot encode them. We need to label encode the features for Bayesian modeling since it looks up the parameters in indexes so I used LabelEncoder from sklearn.preprocessing for that.



Also, I split the data into a training and test set for cross validation (80% of the data in the training set and 20% in the test set).

In [ ]:
# Encoding
df["ForceGroup"] = df["ForceType"].map(force_map)
df["ResistanceGroup"] = df["TypeOfResistance"].replace(resistance_map)
df = df[df["ResistanceGroup"] != "x"]
df['Problem_Group'] = df['Problem'].str.strip().str.lower().map(problem_map).fillna("Welfare & Emergency")

# Label encoding the target variable (force group)
label_force = LabelEncoder()
df["ForceGroup_index"] = label_force.fit_transform(df["ForceGroup"])

# Cross validation (splitting into training and test groups with the test size being 20% of the data and the training group being 80% of the data)
trainset, testset = train_test_split(df, test_size=0.2, random_state=0, stratify=df["ForceGroup_index"])

# Randomly subsampling observations from each force type category to equal the number of observations in the smallest classs (Guns)
# Will help with heavy class imbalance
guns_count = trainset["ForceGroup"].value_counts()["Guns"]
print(f"Number of observations in Guns training set (smallest force type class): {guns_count}")

trainset = (trainset.groupby("ForceGroup", group_keys=False).apply(lambda x: x.sample(n=guns_count, random_state=0)))
print(trainset["ForceGroup"].value_counts())
  # trainset.groupby splits the training set into each force type category. group_keys=False means that it won't add these group labels as an extra index
  # .apply(lamba x: x.sample(n=guns_count, random_state=0): randomly samples the number of observation in "Guns" for each group

X_train = trainset[["Sex", "Race", "ResistanceGroup", "Problem_Group"]]
X_test  = testset[["Sex", "Race", "ResistanceGroup", "Problem_Group"]]
y_train = trainset["ForceGroup_index"]
y_test  = testset["ForceGroup_index"]

# Using one-hot encoder for Sex, Race, and ResistanceGroup (fewer categories)
X_train_encoded = pd.get_dummies(X_train, columns=["Sex", "Race", "ResistanceGroup"], drop_first=True, dtype=int)
X_test_encoded  = pd.get_dummies(X_test,  columns=["Sex", "Race", "ResistanceGroup"], drop_first=True, dtype=int)

# Using target encoder for Problem_Group (since there are more categories so it's hard to one-hot encode)
target_encoder = TargetEncoder()
target_encoder.set_output(transform="pandas")
problem_train = target_encoder.fit_transform(X_train[["Problem_Group"]], y_train)
problem_test  = target_encoder.transform(X_test[["Problem_Group"]])

X_train_final = X_train_encoded.join(problem_train.rename(columns={"Problem_Group": "Problem_Group_encoded"}))
X_test_final  = X_test_encoded.join(problem_test.rename(columns={"Problem_Group": "Problem_Group_encoded"}))

print(f"Training set shape: {X_train_final.shape}") # (number of rows in training set after subsampling, number of feature columns after encoding)
print(f"Test shape: {X_test_final.shape}") # (number of rows in test set, number of feature columns after encoding)

# Label encoding for Bayesian model

label_sex = LabelEncoder()
trainset["Sex_index"] = label_sex.fit_transform(trainset["Sex"])
testset["Sex_index"] = label_sex.transform(testset["Sex"])

label_race = LabelEncoder()
trainset["Race_index"] = label_race.fit_transform(trainset["Race"])
testset["Race_index"] = label_race.transform(testset["Race"])

label_res = LabelEncoder()
trainset["ResistanceGroup_index"] = label_res.fit_transform(trainset["ResistanceGroup"])
testset["ResistanceGroup_index"] = label_res.transform(testset["ResistanceGroup"])

label_problem = LabelEncoder()
trainset["Problem_Group_index"] = label_problem.fit_transform(trainset["Problem_Group"])
testset["Problem_Group_index"] = label_problem.transform(testset["Problem_Group"])

encoders = {
    "ForceGroup": label_force,
    "Sex": label_sex,
    "Race": label_race,
    "ResistanceGroup": label_res,
    "Problem_Group": label_problem,
}



Now, I'm implementing the Bayesian model

In [ ]:
# coordinates used for arviz
coordinates = {
    "obs_id": np.arange(len(trainset)),
    "class": label_force.classes_,
    "sex": label_sex.classes_,
    "race": label_race.classes_,
    "resistance": label_res.classes_,
    "problem": label_problem.classes_,
}

with pm.Model(coords=coordinates) as force_model:
    sex_index = pm.Data("sex_index", trainset["Sex_index"].values, dims="obs_id")
    race_index = pm.Data("race_index", trainset["Race_index"].values, dims="obs_id")
    resistance_index = pm.Data("resistance_index", trainset["ResistanceGroup_index"].values, dims="obs_id")
    problem_index = pm.Data("problem_index", trainset["Problem_Group_index"].values, dims="obs_id")

    # Making an intercept for each group in force
    intercept = pm.StudentT("intercept", nu=4, mu=0, sigma=5, dims="class")

    # Fixed Effects (Demographics) using standard normal as prior distribution
    beta_sex  = pm.Normal("beta_sex",  mu=0, sigma=1, dims=("sex",  "class"))
    beta_race = pm.Normal("beta_race", mu=0, sigma=1, dims=("race", "class"))

    # Noncentered parameterization for resistance type
    sigma_resistance  = pm.HalfNormal("sigma_resistance", sigma=1) # uses half normal as prior since it has to be positive
    offset_resistance = pm.Normal("offset_resistance", 0, 1, dims=("resistance", "class")) # uses normal as prior
    beta_resistance   = pm.Deterministic("beta_resistance", offset_resistance * sigma_resistance, dims=("resistance", "class"))

    # Noncentered parameterization for problem type
    sigma_problem  = pm.HalfNormal("sigma_problem", sigma=1) # again, uses half normal as prior
    offset_problem = pm.Normal("offset_problem", 0, 1, dims=("problem", "class")) # uses normal as prior
    beta_problem   = pm.Deterministic("beta_problem", offset_problem * sigma_problem, dims=("problem", "class"))

    # Indexing into parameter arrays
    mu = (intercept
          + beta_sex[sex_index]
          + beta_race[race_index]
          + beta_resistance[resistance_index]
          + beta_problem[problem_index]
          )

    #Likelihood distribution using softmax
    p = pm.Deterministic("p", pm.math.softmax(mu, axis=1), dims=("obs_id", "class"))
    obs = pm.Categorical("obs", p=p, observed=trainset["ForceGroup_index"].values, dims="obs_id")

    #Sampling
    trace = pm.sample(draws=500, tune=500, chains=2, target_accept=0.95, return_inferencedata=True, random_seed=0)


Checking convergence

To check that the chains have converged use az.summary() to report R-hat (values close to 1.0 indicate good mixing) and effective sample size (ESS).

Also, use az.plot_trace() to make trace plots for each parameter. A well mixed chain will have overlapping lines without any noticeable trends.



In [ ]:
import arviz as az

# R-hat and ESS
print(az.summary(trace, var_names=["intercept", "beta_sex", "beta_race", "sigma_resistance", "sigma_problem"], round_to=3))

# Trace plots
az.plot_trace(trace, var_names=["intercept", "beta_sex", "beta_race", "sigma_resistance", "sigma_problem"])
plt.show()


## Posterior Distribution Histograms

Plot histograms of the estimated posterior distributions of the beta_sex, beta_race, sigma_resistance, and sigma_problem parameters. A weight of 0 would mean that feature has no impact on the predicted force type. If the posterior distribution histogram is centered at 0, it likely is just noise. If it is centered away from 0, that feature has a meaningful influence on the predicted force class.


In [ ]:
beta_sex_samples = np.array(trace.posterior["beta_sex"]).reshape(-1, len(encoders["Sex"].classes_), len(encoders["ForceGroup"].classes_))
beta_race_samples = np.array(trace.posterior["beta_race"]).reshape(-1, len(encoders["Race"].classes_), len(encoders["ForceGroup"].classes_))
beta_resistance_samples = np.array(trace.posterior["beta_resistance"]).reshape(-1, len(encoders["ResistanceGroup"].classes_), len(encoders["ForceGroup"].classes_))
beta_problem_samples    = np.array(trace.posterior["beta_problem"]).reshape(-1, len(encoders["Problem_Group"].classes_), len(encoders["ForceGroup"].classes_))

force_classes = encoders["ForceGroup"].classes_

n_sex = len(encoders["Sex"].classes_)
n_race = len(encoders["Race"].classes_)
n_resistance = len(encoders["ResistanceGroup"].classes_)
n_problem = len(encoders["Problem_Group"].classes_)
n_class = len(force_classes)

colors = ["red", "orange", "green", "blue"]

# Sex and force type histograms
fig, axes = plt.subplots(n_sex, n_class, figsize=(4 * n_class, 3 * n_sex))
axes = axes.ravel()

k = 0
for i in range(n_sex):
    for j in range(n_class):
      samples = beta_sex_samples[:, i, j]
      post_mean = samples.mean()
      axes[k].hist(beta_sex_samples[:, i, j], bins=50, color=colors[j], alpha=0.7)
      axes[k].axvline(0, color="gray", linestyle="--", linewidth=2.0)
      axes[k].axvline(post_mean, color="black", linestyle="-", linewidth=2.0, label=f"mean={post_mean:.3f}")
      axes[k].legend()
      axes[k].set_title(f"{encoders['Sex'].classes_[i]} and {force_classes[j]}")
      k += 1

plt.suptitle("Posterior Distribution Histograms for Sex and Force Type")
plt.tight_layout()
plt.show()




In [ ]:
# Race and force type histograms
fig, axes = plt.subplots(n_race, n_class, figsize=(4 * n_class, 3 * n_race))
axes = axes.ravel()

k = 0
for i in range(n_race):
    for j in range(n_class):
        samples = beta_race_samples[:, i, j]
        post_mean = samples.mean()
        axes[k].hist(beta_race_samples[:, i, j], bins=50, color=colors[j], alpha=0.7)
        axes[k].axvline(0, color="gray", linestyle="--", linewidth=2.0)
        axes[k].axvline(post_mean, color="black", linestyle="-", linewidth=2.0, label=f"mean={post_mean:.3f}")
        axes[k].legend()
        axes[k].set_title(f"{encoders['Race'].classes_[i]} and {force_classes[j]}")
        k += 1

plt.suptitle("Histograms of Posterior Distributions for Race and Force Type")
plt.tight_layout()
plt.show()


In [ ]:
print(trainset["ResistanceGroup"].value_counts())

In [ ]:
fig, axes = plt.subplots(n_resistance, n_class, figsize=(4 * n_class, 3 * n_resistance))
axes = axes.ravel()

k = 0
for i in range(n_resistance):
    for j in range(n_class):
        samples = beta_resistance_samples[:, i, j]
        post_mean = samples.mean()
        axes[k].hist(samples, bins=50, color=colors[j], alpha=0.7)
        axes[k].axvline(0, color="gray", linestyle="--", linewidth=2.0)
        axes[k].axvline(post_mean, color="black", linestyle="-", linewidth=2.0, label=f"mean = {post_mean:.2f}")
        axes[k].legend(fontsize=8)
        axes[k].set_title(f"{encoders["ResistanceGroup"].classes_[i]} and {force_classes[j]}")
        k += 1

plt.suptitle("Posterior Distribution Histograms for Type of Resistance and Force Type", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Problem type and force type histograms
fig, axes = plt.subplots(n_problem, n_class, figsize=(4 * n_class, 3 * n_problem))
axes = axes.ravel()

k = 0
for i in range(n_problem):
    for j in range(n_class):
        samples = beta_problem_samples[:, i, j]
        post_mean = samples.mean()
        axes[k].hist(samples, bins=50, color=colors[j], alpha=0.7)
        axes[k].axvline(0, color="gray", linestyle="--", linewidth=2.0)
        axes[k].axvline(post_mean, color="black", linestyle="-", linewidth= 2.0, label=f"mean = {post_mean:.3f}")
        axes[k].legend(fontsize=8)
        axes[k].set_title(f"{encoders['Problem_Group'].classes_[i]} and {force_classes[j]}")
        k += 1

plt.suptitle("Posterior Distribution Histograms for Problem Type and Force Type", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Histograms for sigma_resistance and sigma_problem
sig2s_resistance = trace.posterior["sigma_resistance"].values.ravel()
sig2s_problem    = trace.posterior["sigma_problem"].values.ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mean_res = sig2s_resistance.mean()
axes[0].hist(sig2s_resistance, bins=50, color="blue", alpha=0.5)
axes[0].axvline(0, color="gray", linestyle="--", linewidth=2.0)
axes[0].axvline(mean_res, color="black", linestyle="-", linewidth=2.0, label=f"mean = {mean_res:.3f}")
axes[0].legend()
axes[0].set_title("Posterior: sigma resistance")

mean_prob = sig2s_problem.mean()
axes[1].hist(sig2s_problem, bins=50, color="blue", alpha=0.7)
axes[1].axvline(0, color="gray", linestyle="--", linewidth=2.0)
axes[1].axvline(mean_prob, color="black", linestyle="-", linewidth=2.0, label=f"mean = {mean_prob:.3f}")
axes[1].legend()
axes[1].set_title("Posterior: sigma problem")

plt.show()


Final calculations: predictions, classification report, RMSE, and sigma^2.

In [ ]:
def predict_multinomial(df_test, trace):
    b_sex = trace.posterior["beta_sex"].mean(dim=["chain", "draw"]).values
    b_race = trace.posterior["beta_race"].mean(dim=["chain", "draw"]).values
    b_resistance = trace.posterior["beta_resistance"].mean(dim=["chain", "draw"]).values
    b_problem = trace.posterior["beta_problem"].mean(dim=["chain", "draw"]).values
    intercept = trace.posterior["intercept"].mean(dim=["chain", "draw"]).values

    # log-odds
    mu_test = (intercept
               + b_sex[df_test["Sex_index"].values]
               + b_race[df_test["Race_index"].values]
               + b_resistance[df_test["ResistanceGroup_index"].values]
               + b_problem[df_test["Problem_Group_index"].values])

    # Making these log-odds into probabilities (taking the e^mu first)
    exp_mu = np.exp(mu_test - np.max(mu_test, axis=1, keepdims=True))
    probabilities = exp_mu / np.sum(exp_mu, axis=1, keepdims=True)
    return np.argmax(probabilities, axis=1)

y_pred = predict_multinomial(testset, trace)
y_true = testset["ForceGroup_index"].values

print("Test set classification:")
print(classification_report(y_true, y_pred, target_names=encoders["ForceGroup"].classes_))

post_RMSE = np.sqrt(mean_squared_error(y_true, y_pred))
post_sigma2 = np.var(y_true - y_pred)

print(f"Posterior mean RMSE = {post_RMSE}")
print(f"Posterior mean sigma^2 = {post_sigma2}")

